<a href="https://colab.research.google.com/github/guneetnagia/LearnAI/blob/RAG/RAG_Hackathon_AST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Hackathon 2: Retrieval-Augmented Generation over AST-1 & AST-2

A question-answering system grounded in two course documents: **AST-1** *(Transitioning from
Research to Production Environment)* and **AST-2** *(Testing the Modules & Packaging)*. It is built
entirely from open-source components: PyMuPDF, LangChain, MTEB embedding models, FAISS and Chroma,
and a Qwen2.5 instruct model running locally, with no API key needed.

```
PDFs ──► clean pages ──► chunks (+page labels) ──► embeddings ──► FAISS / Chroma
                                                                        │
question ──► query embedding ──► similarity or MMR retrieval ◄──────────┘
                                          │
                       numbered passages + grounded prompt ──► LLM ──► cited answer
```

**Every design choice is measured, not assumed.** Retrieval settings are tuned on a separate
`TUNING_SET` of questions, and the final answers are reported on a held-out `EVAL_SET`, so the
reported numbers aren't tuned on.

| Step | Points | Section | What is demonstrated |
|---|---|---|---|
| 1 Load documents | 1 | §1 | PyMuPDF vs pypdf extraction quality, OCR of the screenshots, page metadata |
| 2 Chunking | 1 | §2, §3.2 | recursive / character / token-aware / semantic chunking; size & overlap stats; chunks may cross page breaks and keep correct page labels |
| 3 Embeddings + vector DB | 3 | §3 | 4 open-source MTEB models with correct query/passage prefixes; FAISS and persistent Chroma; retrieval grid scored by context recall, MRR and source accuracy |
| 4 RAG with LLM | 3 | §4 | open-source LLM, grounded prompt with citations, answer self-check, LCEL chain, MMR vs similarity side by side, k × λ sweep |
| 5 Questions | 1 | §5 | 10 logical questions + an out-of-scope probe, automatic answer scoring, answer-level hyperparameter sweep |
| 6 Gradio app | 1 | §6 | streamed, cited answers with the retrieved passages and live retrieval settings |
| 7 HF Spaces | optional | §7 | ready-to-upload Space that runs the configuration chosen here |

**Runtime:** Colab with a T4 GPU (*Runtime → Change runtime type*). On CPU everything still runs,
with a smaller default LLM.

## 0. Setup

In [ ]:
%pip install -q "langchain-core>=1.0,<2" "langchain-text-splitters>=1.0,<2" "langchain-huggingface>=1.0,<2" \
    "langchain-chroma>=1.0,<2" "langchain-community>=0.4,<0.5" "chromadb>=1.0" faiss-cpu \
    "pymupdf>=1.24" pypdf "sentence-transformers>=5.2,<6" "gradio>=6,<7" openai pandas pytesseract pillow

In [ ]:
# Tesseract reads the screenshots (folder trees, pytest and build output) that have no text layer
!apt-get -qq install -y tesseract-ocr > /dev/null

In [ ]:
# ---- Run configuration -------------------------------------------------------------
PDF_PATHS = ["AST-1.pdf", "AST-2.pdf"]  # the knowledge base
LLM_BACKEND = "local"  # "local": open-source model on this machine, no API key needed
#                        "groq" / "hf" / "openai": OpenAI-compatible APIs (key required)
LLM_MODEL = None  # None -> Qwen/Qwen2.5-1.5B-Instruct (0.5B was too weak: 0.58 fact recall,
#                   citation-only replies). "Qwen/Qwen2.5-3B-Instruct" also fits a T4.
OCR_IMAGES = True  # also read the screenshots with Tesseract OCR
EMBED_KEYS = ["minilm", "bge-small", "bge-base", "e5-base"]  # compared in Step 3
RUN_ANSWER_GRID = True  # Step 5 answer-level sweep through the LLM (~5 min on a T4)
CHROMA_DIR = "chroma_ast"  # persistent vector database folder

# API keys are only needed for the API backends, e.g. with Colab secrets:
# import os; from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
import shutil
from pathlib import Path

# Move AST PDFs from sample_data to current workspace if they exist there
for pdf in ['AST-1.pdf', 'AST-2.pdf']:
    src = Path('/content/sample_data') / pdf
    dst = Path('.') / pdf
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f"Copied {pdf} to working directory.")


In [ ]:
import glob
import shutil
from pathlib import Path

# Dynamically search the workspace for AST-1.pdf and AST-2.pdf
all_files = glob.glob('**/AST-*.pdf', recursive=True) + glob.glob('**/AST-*.PDF', recursive=True)
print("Found PDF files in workspace:", all_files)

for file_path_str in all_files:
    src_path = Path(file_path_str)
    dest_path = Path('.') / src_path.name
    if src_path.resolve() != dest_path.resolve():
        shutil.copy(src_path, dest_path)
        print(f"Successfully copied {src_path.name} to the working directory!")

Found PDF files in workspace: []


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
missing = [p for p in PDF_PATHS if not Path(p).exists()]
if missing and IN_COLAB:
    from google.colab import files

    print("Please upload:", ", ".join(missing))
    uploaded = files.upload()

missing = [p for p in PDF_PATHS if not Path(p).exists()]
assert not missing, f"Missing PDFs {missing}: place them next to this notebook."
print("Knowledge base files successfully loaded:", PDF_PATHS)

Please upload: AST-1.pdf, AST-2.pdf


KeyboardInterrupt: 

In [ ]:
import shutil
from pathlib import Path

# Source directory where AST-1.pdf and AST-2.pdf are stored
source_dir = Path('/content/sample_data/.ipynb_checkpoints/data')

for pdf in ['AST-1.pdf', 'AST-2.pdf']:
    src = source_dir / pdf
    dst = Path('.') / pdf
    if src.exists():
        shutil.copy(src, dst)
        print(f"Successfully copied {pdf} to working directory.")
    else:
        print(f"Could not find {pdf} in {source_dir}")

Successfully copied AST-1.pdf to working directory.
Successfully copied AST-2.pdf to working directory.


In [ ]:
from __future__ import annotations

import bisect
import os
import re
import threading
import uuid
import warnings
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Callable, Iterable, Iterator, Sequence

import numpy as np
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

import warnings
from dataclasses import replace

import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 110)

## 1. Load the documents

Each PDF is read page by page into LangChain `Document`s carrying `source`, `doc_name` and `page`.
Only whitespace is normalised and the wording is untouched. Two extractors are compared because an
extraction error (a word split in two) silently hurts embedding, retrieval and the final answer.

Both guides also contain **screenshots with no text layer**: folder trees, the `pytest` run and the
package build log. Facts such as *"collected 2 items … 2 passed"* or the exact wheel file name exist
only there. With `OCR_IMAGES = True`, each screenshot is rendered at 300 dpi, inverted if it is a dark
terminal, read with Tesseract, cleaned of tree icons, and appended to its page under
`[Screenshot text (OCR)]`, so it is chunked, cited and shown like any other text.

In [ ]:
def clean_text(text: str) -> str:
    """Normalise whitespace from PDF extraction without changing the wording."""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)  # collapse runs of spaces
    text = re.sub(r" *\n *", "\n", text)  # trim spaces around line breaks
    text = re.sub(r"\n{3,}", "\n\n", text)  # at most one blank line
    return text.strip()


_OCR_LEAD = re.compile(r"^(?:[^\w\s]+|[vVJ4]\b|\d\)|_(?=\s)|\s)+")
OCR_MARKER = "[Screenshot text (OCR)]"


def _clean_ocr(text: str) -> str:
    """Drop folder-tree icons / arrows that OCR reads as symbols, and lines with no real word."""
    lines = []
    for raw in text.splitlines():
        line = _OCR_LEAD.sub("", raw).strip()
        if re.search(r"\w{3,}", line):
            lines.append(line)
    return "\n".join(lines)


def ocr_available() -> bool:
    try:
        import pytesseract

        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False


def _ocr_page_images(page, dpi: int = 300, min_width: float = 40, min_height: float = 8) -> str:
    """OCR every screenshot on a PyMuPDF page (folder trees, terminal output)."""
    import io

    import pytesseract
    from PIL import Image, ImageOps, ImageStat

    texts, seen = [], set()
    for image_info in page.get_images(full=True):
        for rect in page.get_image_rects(image_info[0]):
            key = tuple(round(v) for v in rect)
            if rect.width < min_width or rect.height < min_height or key in seen:
                continue  # skip icons and duplicate placements
            seen.add(key)
            image = Image.open(io.BytesIO(page.get_pixmap(clip=rect, dpi=dpi).tobytes("png"))).convert("L")
            if ImageStat.Stat(image).mean[0] < 110:  # terminal screenshot: light text on dark
                image = ImageOps.invert(image)
            text = _clean_ocr(pytesseract.image_to_string(image))
            if text:
                texts.append(text)
    return "\n\n".join(texts)


def _extract_pages(path: Path, loader: str, ocr_images: bool = False) -> list[str]:
    if loader == "pymupdf":
        import pymupdf

        with pymupdf.open(path) as pdf:
            pages = []
            for page in pdf:
                text = page.get_text()
                if ocr_images:
                    ocr = _ocr_page_images(page)
                    if ocr:
                        text += f"\n\n{OCR_MARKER}\n{ocr}"
                pages.append(text)
            return pages
    if loader == "pypdf":
        if ocr_images:
            raise ValueError("OCR of screenshots needs loader='pymupdf'.")
        from pypdf import PdfReader

        return [page.extract_text() or "" for page in PdfReader(path).pages]
    raise ValueError(f"Unknown loader {loader!r}; use 'pymupdf' or 'pypdf'.")


def load_pdfs(
    paths: Iterable[str | Path], loader: str = "pymupdf", ocr_images: bool = False
) -> list[Document]:
    """Load PDFs into one LangChain Document per page with small, scalar-only metadata.

    ``loader="pymupdf"`` (default) extracts these PDFs cleanly; ``"pypdf"`` is kept
    for comparison because it splits words ("c orresponding", "Pytes t").
    ``ocr_images=True`` also reads the screenshots (folder trees, pytest and build
    output), which have no text layer. Needs the Tesseract binary; skipped with a
    warning when it is missing.
    """
    if ocr_images and not ocr_available():
        warnings.warn("Tesseract OCR not found: screenshots are skipped (apt-get install tesseract-ocr).")
        ocr_images = False
    pages: list[Document] = []
    for path in map(Path, paths):
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {path}")
        for number, raw in enumerate(_extract_pages(path, loader, ocr_images), start=1):
            text = clean_text(raw)
            if text:
                metadata = {"source": path.name, "doc_name": path.stem, "page": number}
                pages.append(Document(page_content=text, metadata=metadata))
    return pages

In [ ]:
import sys
from pathlib import Path

# Check if PDFs are missing and prompt for upload if in Google Colab
missing = [p for p in PDF_PATHS if not Path(p).exists()]
if missing and "google.colab" in sys.modules:
    from google.colab import files
    print("Please upload the missing PDF files:", ", ".join(missing))
    uploaded = files.upload()

text_pages = load_pdfs(PDF_PATHS)  # PyMuPDF text layer only
pages_pypdf = load_pdfs(PDF_PATHS, loader="pypdf")
pages = load_pdfs(PDF_PATHS, ocr_images=OCR_IMAGES)  # what the system uses

SPLIT_WORDS = ["c orresponding", "effect ively", "Pytes t", "pre -pipeline", "sub -folders", "pre -processing"]


def split_word_count(docs):
    text = " ".join(d.page_content for d in docs)
    return sum(text.count(w) for w in SPLIT_WORDS)


display(
    pd.DataFrame(
        [
            {
                "loader": name,
                "pages": len(docs),
                "characters": sum(len(d.page_content) for d in docs),
                "split-word artifacts": split_word_count(docs),
            }
            for name, docs in [("pypdf", pages_pypdf), ("PyMuPDF", text_pages), ("PyMuPDF + OCR", pages)]
        ]
    )
)
display(pd.DataFrame([{**p.metadata, "characters": len(p.page_content)} for p in pages]))
print(pages[0].page_content[:500])

for p in pages:
    if OCR_MARKER in p.page_content:
        print(f"\n--- OCR text from {p.metadata['doc_name']} p.{p.metadata['page']} ---")
        print(p.page_content.split(OCR_MARKER, 1)[1].strip()[:400])

,loader,pages,characters,split-word artifacts
0,pypdf,11,15215,7
1,PyMuPDF,11,15206,0
2,PyMuPDF + OCR,11,16481,0


,source,doc_name,page,characters
0,AST-1.pdf,AST-1,1,1823
1,AST-1.pdf,AST-1,2,895
2,AST-1.pdf,AST-1,3,1936
3,AST-1.pdf,AST-1,4,1809
4,AST-1.pdf,AST-1,5,182
5,AST-2.pdf,AST-2,1,2029
6,AST-2.pdf,AST-2,2,1908
7,AST-2.pdf,AST-2,3,1459
8,AST-2.pdf,AST-2,4,2125
9,AST-2.pdf,AST-2,5,1609


Module 3: AST-1

TITLE: Transitioning from Research to Production Environment
LEARNING OBJECTIVES:
At the end of the experiment, you will be able to transition from the research environment to the
production environment. You will understand the concept of modularization and convert the model
developed in Jupyter Notebook into different modules tailored to specific functionalities: Data
Manager, Training, Pipeline, Predict, etc.

You will be able to understand and implement the following aspects:

--- OCR text from AST-1 p.2 ---
project
requirements
requirements.txt
titanic_model
config
datasets
processing
trained_models
config.yml
pipeline.py
predict.py
train_pipeline.py
VERSION
_init_.py

project
requirements
titanic_model

--- OCR text from AST-1 p.3 ---
project
requirements
requirements.txt
titanic_model
config
core.py
_init_.py
datasets
titanic.csv
_init_.py
processing
data_manager.py
features.py
validation.py
_init_.py
trained_models
_init_.py
config.yml
pipeline.py
predict.py
tra

**Finding.** pypdf breaks several words in these PDFs (e.g. *"c orresponding"*, *"Pytes t"*), while
PyMuPDF extracts them cleanly, so PyMuPDF is used from here on. OCR adds the content of the screenshots;
the terminal output reads cleanly, and the folder trees keep a little icon noise (e.g. `_init_.py`).

## 2. Split the documents into chunks

Four chunking strategies are implemented:

* **recursive**: `RecursiveCharacterTextSplitter`, which splits on paragraphs, then lines, sentences and words.
* **character**: fixed-size `CharacterTextSplitter` on line breaks, as a baseline.
* **token**: recursive splitting measured in the embedding model's *own tokens*. Embedders truncate
  long inputs (all-MiniLM-L6-v2 reads only 256 tokens, bge/e5 read 512), so measuring in tokens
  guarantees nothing is silently cut off.
* **semantic**: embeds every sentence with its neighbours and starts a new chunk where the meaning
  shifts (implemented here because `langchain-experimental`, which held `SemanticChunker`, is being
  sunset). It needs an embedding model, so it is run in §3.2.

Pages of each PDF are **merged before splitting**: AST-2's procedure, for example, runs from page 5 onto
page 6, so splitting page by page would cut it in half. Each chunk's character offset is mapped back
to a page range (`page`, `page_end`) for citations. An optional **contextual header**
(`[AST-2: Testing the Modules & Packaging]`) can be prepended so the embedding knows which guide a
passage belongs to; §3 measures whether it helps.

**Overlap is guaranteed at every boundary.** LangChain's splitter only carries over whole paragraphs or
lines that fit in the overlap budget; on these PDFs, 39% of the boundaries in the 800/120 setting had no
overlap at all. So chunks are first cut at `chunk_size − chunk_overlap` with no overlap, then each one
is extended back into the previous chunk by up to `chunk_overlap`, starting at a word. Every boundary
overlaps, and no chunk is longer than `chunk_size`. The table below checks this for each setting.

In [ ]:
def merge_pages(pages: Sequence[Document]) -> list[Document]:
    """Join each PDF's pages into a single Document so chunks can cross page breaks.

    Page start offsets are kept in private metadata and used later to label every
    chunk with the page(s) it came from.
    """
    by_source: dict[str, list[Document]] = {}
    for page in pages:
        by_source.setdefault(page.metadata["source"], []).append(page)

    merged = []
    for source, doc_pages in by_source.items():
        text, starts, numbers = "", [], []
        for page in sorted(doc_pages, key=lambda d: d.metadata["page"]):
            if text:
                text += "\n\n"
            starts.append(len(text))
            numbers.append(page.metadata["page"])
            text += page.page_content
        title = re.search(r"TITLE:\s*(.+)", text)
        merged.append(
            Document(
                page_content=text,
                metadata={
                    "source": source,
                    "doc_name": doc_pages[0].metadata["doc_name"],
                    "title": title.group(1).strip() if title else "",
                    "_page_starts": starts,
                    "_page_numbers": numbers,
                },
            )
        )
    return merged


_SENTENCE_RE = re.compile(r"[^\n.!?]*(?:[.!?](?!\w)|\n|$)")


def _sentence_spans(text: str) -> list[tuple[int, int]]:
    """(start, end) offsets of sentences / lines, ignoring empty ones."""
    spans = []
    for m in _SENTENCE_RE.finditer(text):
        if m.group().strip():
            spans.append((m.start(), m.end()))
    return spans


def semantic_split(
    doc: Document,
    embeddings: Embeddings,
    breakpoint_percentile: float = 90,
    buffer: int = 1,
    max_chars: int = 1200,
    min_chars: int = 150,
) -> list[Document]:
    """Semantic chunking: cut where the meaning shifts between neighbouring sentences.

    Each sentence is embedded together with ``buffer`` neighbours; a new chunk starts
    where the cosine distance to the next window is above the given percentile
    (or when the chunk would exceed ``max_chars``).
    """
    text = doc.page_content
    spans = _sentence_spans(text)
    if len(spans) < 2:
        return [Document(page_content=text.strip(), metadata={**doc.metadata, "start_index": 0})]

    sentences = [text[s:e].strip() for s, e in spans]
    windows = [" ".join(sentences[max(0, i - buffer) : i + buffer + 1]) for i in range(len(sentences))]
    vecs = np.asarray(embeddings.embed_documents(windows), dtype=float)
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
    distances = 1.0 - np.sum(vecs[:-1] * vecs[1:], axis=1)
    threshold = np.percentile(distances, breakpoint_percentile)

    chunks, group_start = [], spans[0][0]
    for i in range(1, len(spans)):
        current_len = spans[i - 1][1] - group_start
        topic_shift = distances[i - 1] > threshold and current_len >= min_chars
        too_long = spans[i][1] - group_start > max_chars
        if topic_shift or too_long:
            chunks.append((group_start, spans[i - 1][1]))
            group_start = spans[i][0]
    chunks.append((group_start, spans[-1][1]))

    out = []
    for start, end in chunks:
        raw = text[start:end]
        stripped = raw.strip()
        if stripped:
            offset = start + (len(raw) - len(raw.lstrip()))
            out.append(Document(page_content=stripped, metadata={**doc.metadata, "start_index": offset}))
    return out


def _label_pages(chunk: Document) -> Document:
    """Replace private offset metadata with page / page_end (scalars only)."""
    meta = dict(chunk.metadata)
    starts = meta.pop("_page_starts")
    numbers = meta.pop("_page_numbers")
    start = meta.get("start_index", 0)
    end = start + max(len(chunk.page_content) - 1, 0)
    meta["page"] = numbers[bisect.bisect_right(starts, start) - 1]
    meta["page_end"] = numbers[bisect.bisect_right(starts, end) - 1]
    return Document(page_content=chunk.page_content, metadata=meta)


def _locate(text: str, pieces: Sequence[str]) -> list[int]:
    """Character offset of each (in-order) piece inside ``text``."""
    offsets, cursor = [], 0
    for piece in pieces:
        idx = text.find(piece, cursor)
        if idx == -1:
            idx = text.find(piece)
        offsets.append(idx)
        cursor = idx + 1
    return offsets


def _add_overlap(text: str, spans: list[tuple[int, int]], overlap: int, length=len) -> list[tuple[int, int]]:
    """Extend each chunk back into the previous one by up to ``overlap`` (as measured by
    ``length``), starting at a word boundary, so *every* boundary overlaps.

    LangChain's own overlap only carries whole pieces (paragraphs / lines) that fit in the
    overlap budget, so a long last paragraph leaves the next chunk with no overlap at all.
    """
    if overlap <= 0 or len(spans) < 2:
        return spans
    out = [spans[0]]
    for (prev_start, _), (start, end) in zip(spans, spans[1:]):
        new_start = start
        for gap in reversed(list(re.finditer(r"\s+", text[prev_start:start]))):
            candidate = prev_start + gap.end()
            if candidate >= start:
                continue  # the whitespace right at the boundary adds nothing
            if length(text[candidate:start]) > overlap:
                break
            new_start = candidate
        out.append((new_start, end))
    return out


def _make_splitter(method: str, chunk_size: int, chunk_overlap: int, tokenizer=None):
    common = dict(chunk_size=chunk_size, chunk_overlap=chunk_overlap, keep_separator="end")
    if method == "recursive":
        return RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", " ", ""], **common)
    if method == "character":
        return CharacterTextSplitter(separator="\n", **common)
    if method == "token":
        if tokenizer is None:
            raise ValueError("token chunking needs a `tokenizer` with .tokenize().")
        return RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=lambda t: len(tokenizer.tokenize(t)),
            **common,
        )
    raise ValueError(f"Unknown chunking method {method!r}.")


def chunk_header(doc: Document) -> str:
    title = doc.metadata.get("title")
    return f"[{doc.metadata['doc_name']}: {title}]" if title else f"[{doc.metadata['doc_name']}]"


def chunk_body(doc: Document) -> str:
    """Chunk text without the contextual header (for display)."""
    return doc.page_content[doc.metadata.get("header_len", 0) :]


def page_label(doc: Document) -> str:
    start, end = doc.metadata.get("page"), doc.metadata.get("page_end")
    return f"p.{start}" if end in (None, start) else f"pp.{start}-{end}"


def split_documents(
    pages: Sequence[Document],
    method: str = "recursive",
    chunk_size: int = 800,
    chunk_overlap: int = 120,
    add_headers: bool = False,
    embeddings: Embeddings | None = None,
    tokenizer=None,
    **semantic_kwargs,
) -> list[Document]:
    """Split loaded pages into chunks labelled with document, page range and offset.

    method:
        "recursive"  RecursiveCharacterTextSplitter (paragraph > line > sentence > word)
        "character"  CharacterTextSplitter on line breaks (fixed-size baseline)
        "token"      recursive splitting measured in *embedding-model tokens*, so no chunk
                     is silently truncated by the embedder (needs ``tokenizer``)
        "semantic"   embedding-based topic-shift chunking (needs ``embeddings``)
    add_headers: prefix every chunk with "[AST-x: <title>]" (contextual chunk header)
        so the embedding knows which guide a passage belongs to.
    chunk_overlap: every chunk (after the first) starts with up to this much of the
        previous chunk, so a fact on a boundary is whole in at least one chunk. Chunks
        never exceed chunk_size (semantic: max_chars) including the overlap.
    """
    if not 0 <= chunk_overlap < chunk_size:
        raise ValueError("chunk_overlap must be >= 0 and smaller than chunk_size.")
    length = (lambda t: len(tokenizer.tokenize(t))) if method == "token" and tokenizer is not None else len
    raw_chunks: list[Document] = []
    for doc in merge_pages(pages):
        text = doc.page_content
        # 1) cut chunks of (size - overlap) with no overlap of their own ...
        if method == "semantic":
            if embeddings is None:
                raise ValueError("semantic chunking needs `embeddings`.")
            kwargs = {**semantic_kwargs, "max_chars": semantic_kwargs.get("max_chars", 1200) - chunk_overlap}
            pieces = [c.page_content for c in semantic_split(doc, embeddings, **kwargs)]
        else:
            pieces = _make_splitter(method, chunk_size - chunk_overlap, 0, tokenizer).split_text(text)
        spans = [(start, start + len(piece)) for piece, start in zip(pieces, _locate(text, pieces))]
        # 2) ... then give every chunk an explicit overlap with the one before it
        for start, end in _add_overlap(text, spans, chunk_overlap, length):
            raw_chunks.append(
                Document(page_content=text[start:end], metadata={**doc.metadata, "start_index": start})
            )

    chunks = []
    for i, raw in enumerate(raw_chunks):
        chunk = _label_pages(raw)
        chunk.metadata["chunk_id"] = i
        if add_headers:
            header = chunk_header(chunk) + "\n"
            chunk.page_content = header + chunk.page_content
            chunk.metadata["header_len"] = len(header)
        chunks.append(chunk)
    return chunks

In [ ]:
from transformers import AutoTokenizer

# Tokenizer of the default embedding model: used for token-aware chunking and for token counts.
bge_tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

CHUNK_CONFIGS = {
    "recursive-500/75": dict(method="recursive", chunk_size=500, chunk_overlap=75),
    "recursive-800/120": dict(method="recursive", chunk_size=800, chunk_overlap=120),
    "recursive-1200/180": dict(method="recursive", chunk_size=1200, chunk_overlap=180),
    "recursive-800/0": dict(method="recursive", chunk_size=800, chunk_overlap=0),
    "character-800/120": dict(method="character", chunk_size=800, chunk_overlap=120),
    "token-160/32": dict(method="token", chunk_size=160, chunk_overlap=32),
}


def overlaps_of(chunks):
    # characters each chunk shares with the one before it (same PDF only)
    return [
        a.metadata["start_index"] + len(chunk_body(a)) - b.metadata["start_index"]
        for a, b in zip(chunks, chunks[1:])
        if a.metadata["source"] == b.metadata["source"]
    ]


def chunk_stats(name, chunks, tokenizer=bge_tokenizer):
    lengths = [len(c.page_content) for c in chunks]
    tokens = [len(tokenizer.tokenize(c.page_content)) for c in chunks]
    return {
        "config": name,
        "chunks": len(chunks),
        "avg chars": round(float(np.mean(lengths))),
        "max chars": max(lengths),
        "max tokens": max(tokens),
        "chunks > 256 tokens (MiniLM truncates)": sum(t > 256 for t in tokens),
        "cross-page chunks": sum(c.metadata["page_end"] > c.metadata["page"] for c in chunks),
        "boundaries without overlap": sum(o <= 0 for o in overlaps_of(chunks)),
        "mean overlap (chars)": round(float(np.mean(overlaps_of(chunks)))),
    }


chunk_sets = {name: split_documents(pages, tokenizer=bge_tokenizer, **cfg) for name, cfg in CHUNK_CONFIGS.items()}
display(pd.DataFrame([chunk_stats(name, chunks) for name, chunks in chunk_sets.items()]))

example = chunk_sets["recursive-800/120"][5]
print(example.metadata)
print(example.page_content)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

,config,chunks,avg chars,max chars,max tokens,chunks > 256 tokens (MiniLM truncates),cross-page chunks,boundaries without overlap,mean overlap (chars)
0,recursive-500/75,57,354,493,159,0,11,0,67
1,recursive-800/120,34,592,796,239,0,12,0,113
2,recursive-1200/180,21,942,1190,281,6,11,0,172
3,recursive-800/0,29,567,799,232,0,7,27,-2
4,character-800/120,26,742,798,242,0,12,0,116
5,token-160/32,39,553,873,160,0,13,0,137


{'source': 'AST-1.pdf', 'doc_name': 'AST-1', 'title': 'Transitioning from Research to Production Environment', 'start_index': 2379, 'page': 2, 'page_end': 3, 'chunk_id': 5}
-

1) Reliability/Reproducibility
2) Maintainability & Adaptability
3) Testability
4) Scalability
5) Version control

[Screenshot text (OCR)]
project
requirements
requirements.txt
titanic_model
config
datasets
processing
trained_models
config.yml
pipeline.py
predict.py
train_pipeline.py
VERSION
_init_.py

project
requirements
titanic_model

Understanding the functionality of each file and folder

Given below is the project folder structure showing all files and sub-folders. In the explanation,
the numberings are given to show how each component is connected one after the other in a logical
sequence, and the explanation is written in front of the corresponding file in the folder structure
for ease of traceability of each file location.

1. requirements: Dependencies-libraries, packages.


Larger chunks carry more context per passage but blur retrieval, and the largest exceed MiniLM's
256-token window, so that model never sees their tail. Overlap protects facts that straddle a
boundary. Which combination answers questions best is measured in §3.3.

## 3. Embeddings and vector databases

### 3.1 Open-source embedding models from the MTEB leaderboard

| key | model | params | dim | query / passage prefix |
|---|---|---|---|---|
| `minilm` | sentence-transformers/all-MiniLM-L6-v2 | 22M | 384 | none |
| `bge-small` | BAAI/bge-small-en-v1.5 | 33M | 384 | retrieval instruction on queries |
| `bge-base` | BAAI/bge-base-en-v1.5 | 109M | 768 | retrieval instruction on queries |
| `e5-base` | intfloat/e5-base-v2 | 109M | 768 | `query:` / `passage:` |

bge and e5 were trained with these prefixes, and leaving them out is a common, silent loss of accuracy.
`PrefixedEmbeddings` adds them automatically. Vectors are L2-normalised, so the inner product
equals cosine similarity.

In [ ]:
# Open-source embedding models from the MTEB leaderboard. bge / e5 are trained with
# query & passage prefixes; using them is worth several points of retrieval accuracy.
EMBEDDING_MODELS: dict[str, dict[str, str]] = {
    "minilm": {
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "",
        "doc_prefix": "",
    },
    "bge-small": {
        "model_name": "BAAI/bge-small-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "doc_prefix": "",
    },
    "bge-base": {
        "model_name": "BAAI/bge-base-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "doc_prefix": "",
    },
    "e5-base": {
        "model_name": "intfloat/e5-base-v2",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
}


class PrefixedEmbeddings(Embeddings):
    """Adds the model-specific query / passage prefixes around any LangChain embedder."""

    def __init__(self, base: Embeddings, query_prefix: str = "", doc_prefix: str = ""):
        self.base = base
        self.query_prefix = query_prefix
        self.doc_prefix = doc_prefix

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.base.embed_documents([self.doc_prefix + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return self.base.embed_query(self.query_prefix + text)


def _cuda_available() -> bool:
    try:
        import torch

        return bool(torch.cuda.is_available())
    except ImportError:
        return False


@lru_cache(maxsize=8)
def get_embeddings(key: str = "bge-small", device: str | None = None) -> Embeddings:
    """Load (once) an open-source sentence-transformers model with normalised vectors."""
    from langchain_huggingface import HuggingFaceEmbeddings

    spec = EMBEDDING_MODELS[key]
    base = HuggingFaceEmbeddings(
        model_name=spec["model_name"],
        model_kwargs={"device": device or ("cuda" if _cuda_available() else "cpu")},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    )
    return PrefixedEmbeddings(base, spec["query_prefix"], spec["doc_prefix"])

In [ ]:
emb = get_embeddings("bge-small")
query = emb.embed_query("Which file defines the pytest fixture?")
for passage in ["conftest.py: Here we are defining a fixture for pytest.", "VERSION: Contains the version tag."]:
    print(f"cosine {np.dot(query, emb.embed_documents([passage])[0]):.3f}  {passage}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cosine 0.860  conftest.py: Here we are defining a fixture for pytest.
cosine 0.517  VERSION: Contains the version tag.


### 3.2 Semantic chunking (needs the embedder, so it runs here)

In [ ]:
CHUNK_CONFIGS["semantic-p90"] = dict(method="semantic", breakpoint_percentile=90, max_chars=1200)
chunk_sets["semantic-p90"] = split_documents(pages, embeddings=emb, **CHUNK_CONFIGS["semantic-p90"])
display(pd.DataFrame([chunk_stats("semantic-p90", chunk_sets["semantic-p90"])]))

,config,chunks,avg chars,max chars,max tokens,chunks > 256 tokens (MiniLM truncates),cross-page chunks,boundaries without overlap,mean overlap (chars)
0,semantic-p90,31,636,1196,263,1,10,0,111


### 3.3 Vector databases (FAISS and Chroma) and retrievers

**FAISS** is an in-memory index, used for the fast experiment grid below. **Chroma** is persisted to disk
for the final system. Rebuilding a persisted collection replaces it, so re-running a cell never
duplicates data. `get_retriever` turns either store into a similarity or MMR retriever (MMR is
examined in detail in §4).

In [ ]:
def build_vectorstore(
    chunks: Sequence[Document],
    embeddings: Embeddings,
    backend: str = "faiss",
    persist_directory: str | None = None,
    collection_name: str | None = None,
):
    """Embed chunks and index them.

    backend="faiss"   in-memory, fastest for experiments
    backend="chroma"  optionally persisted to disk; an existing collection with the
                      same name is replaced so re-running a cell never duplicates data
    """
    if backend == "faiss":
        from langchain_community.vectorstores import FAISS

        return FAISS.from_documents(list(chunks), embeddings)
    if backend == "chroma":
        from langchain_chroma import Chroma

        name = collection_name or f"ast-{uuid.uuid4().hex[:8]}"
        if persist_directory:
            Chroma(collection_name=name, persist_directory=persist_directory).delete_collection()
        return Chroma.from_documents(
            list(chunks),
            embeddings,
            collection_name=name,
            persist_directory=persist_directory,
            collection_metadata={"hnsw:space": "cosine"},
        )
    raise ValueError(f"Unknown backend {backend!r}; use 'faiss' or 'chroma'.")


def get_retriever(
    vectorstore, search_type: str = "mmr", k: int = 6, fetch_k: int = 20, lambda_mult: float = 0.75
):
    """Similarity search returns the k closest chunks; MMR re-ranks ``fetch_k`` candidates
    to balance relevance (``lambda_mult`` -> 1) against diversity (-> 0).

    Defaults k=6, lambda_mult=0.75: full recall on these short guides with less repetition
    than plain similarity search (see the k x lambda sweep in the notebook)."""
    if search_type == "similarity":
        return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": max(fetch_k, k), "lambda_mult": lambda_mult},
        )
    raise ValueError(f"Unknown search_type {search_type!r}; use 'similarity' or 'mmr'.")

In [ ]:
chroma_store = build_vectorstore(
    chunk_sets["recursive-800/120"], emb, backend="chroma", persist_directory=CHROMA_DIR, collection_name="ast_docs"
)
print("Chroma collection size:", chroma_store._collection.count(), "chunks persisted in", CHROMA_DIR)
for doc, distance in chroma_store.similarity_search_with_score("How do I build the package?", k=3):
    print(f"cosine distance {distance:.3f} | {doc.metadata['doc_name']} {page_label(doc)} | {chunk_body(doc)[:80]!r}")

Chroma collection size: 34 chunks persisted in chroma_ast
cosine distance 0.269 | AST-2 p.5 | 'folder in VS code\n\n2. Inside the project_with_test_and_package, create a virtual'
cosine distance 0.279 | AST-2 pp.5-6 | 'and the model is working as intended\n\n6. For building the package from the model'
cosine distance 0.279 | AST-2 p.6 | '--upgrade build\n\nAnd then\nRun: ...> py -m build\n\nYou will see result like this u'


### 3.4 How retrieval is scored

Each tuning question lists the facts a correct answer needs (e.g. *"How do I create the virtual
environment?"* needs `python -m venv`, `activate` and the interpreter selection). For a retriever:

* **context recall** is the share of needed facts present in the top-k passages, so the LLM *can* answer;
* **MRR** is the reciprocal rank of the first passage containing a fact, rewarding good ranking;
* **top-1 source accuracy** checks whether the best passage comes from the right guide (AST-1 vs AST-2).

In [ ]:
@dataclass
class EvalItem:
    """A question plus the facts a good answer must contain.

    ``facts`` is a list of fact groups; each group lists acceptable phrasings
    (case-insensitive). ``source`` is the expected document, or None when the
    question is out of scope and the system should decline.
    """

    question: str
    facts: list[list[str]]
    source: str | None


_REFUSAL_MARKERS = (
    "couldn't find",
    "could not find",
    "not in the",
    "don't know",
    "do not know",
    "not mentioned",
)


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", text).lower()


def fact_recall(text: str, facts: Sequence[Sequence[str]]) -> float:
    """Fraction of fact groups with at least one phrasing present in ``text``."""
    if not facts:
        return 1.0
    t = _norm(text)
    return sum(any(_norm(alt) in t for alt in group) for group in facts) / len(facts)


def is_refusal(answer: str) -> bool:
    return any(m in answer.lower() for m in _REFUSAL_MARKERS)


def evaluate_retrieval(retrieve: Callable[[str], list[Document]], eval_set: Sequence[EvalItem]) -> dict:
    """Context recall, MRR of the first chunk holding a fact, and top-1 source accuracy."""
    recalls, rr, source_hits = [], [], []
    for item in eval_set:
        if item.source is None:
            continue
        docs = retrieve(item.question)
        recalls.append(fact_recall(" ".join(d.page_content for d in docs), item.facts))
        rank = next((i for i, d in enumerate(docs, 1) if fact_recall(d.page_content, item.facts) > 0), None)
        rr.append(1.0 / rank if rank else 0.0)
        source_hits.append(bool(docs) and docs[0].metadata.get("doc_name") == item.source)
    return {
        "context_recall": float(np.mean(recalls)),
        "mrr": float(np.mean(rr)),
        "top1_source_acc": float(np.mean(source_hits)),
    }


def context_diversity(docs: Sequence[Document], embeddings: Embeddings) -> float:
    """Mean pairwise cosine similarity of retrieved chunks (lower = more diverse)."""
    if len(docs) < 2:
        return 0.0
    v = np.asarray(embeddings.embed_documents([d.page_content for d in docs]), dtype=float)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    sims = v @ v.T
    n = len(docs)
    return float((sims.sum() - n) / (n * (n - 1)))


# Questions used ONLY to choose hyperparameters (Steps 2-5). The final report in
# Step 5 uses the separate EVAL_SET, so reported numbers are not tuned on.
TUNING_SET: list[EvalItem] = [
    EvalItem("Which file reads config.yml and sets up the dataset and model paths?", [["core.py"]], "AST-1"),
    EvalItem("What does predict.py do?", [["inference"], ["saved model"]], "AST-1"),
    EvalItem("Where is the trained model stored?", [["trained_models"]], "AST-1"),
    EvalItem("What is validation.py used for?", [["validate"], ["predict.py"]], "AST-1"),
    EvalItem("Why is testing ML systems harder than testing traditional software?", [["data"]], "AST-2"),
    EvalItem("What does the MANIFEST.in file make sure is included?", [["pickle"], ["csv"]], "AST-2"),
    EvalItem(
        "Which requirements file is installed before running the tests?", [["test_requirements.txt"]], "AST-2"
    ),
    EvalItem("What features does pytest provide?", [["fixture"], ["test discovery"]], "AST-2"),
    EvalItem("What information does setup.py contain?", [["metadata"], ["version"]], "AST-2"),
    EvalItem("Which build dependencies are listed in pyproject.toml?", [["setuptools"], ["wheel"]], "AST-2"),
    # Harder, single-fact questions: with only 30 chunks, broad questions let every
    # configuration score 1.0, so these make the grid separate good settings from great ones.
    EvalItem(
        "What should you do if creating or activating the virtual environment fails?",
        [["environment setup document"]],
        "AST-1",
    ),
    EvalItem(
        "Which transformation class does the sample test in test_features.py check?",
        [["age_col_tfr"]],
        "AST-2",
    ),
    EvalItem(
        "How is production code usually prepared for vertical or horizontal scaling?",
        [["containerization"]],
        "AST-1",
    ),
    EvalItem(
        "Which coding standard should production code follow so others can read it?", [["pep8"]], "AST-1"
    ),
    EvalItem(
        "Which benefits of software testing does AST-2 list?",
        [["error detection"], ["risk mitigation"], ["refactoring"]],
        "AST-2",
    ),
    EvalItem("Where does setup.py get the package version from?", [["version file"]], "AST-2"),
    EvalItem(
        "Which notebook shows how the test cases in test_features.py are written?",
        [["experimentation_phase_2_with_test"]],
        "AST-2",
    ),
    EvalItem("Who wrote the novel Pride and Prejudice?", [], None),
]

### 3.5 Experiment grid: embedding model × chunking × header × search type

4 models × 7 chunkings × header on/off × similarity/MMR = 112 retrieval configurations, all scored on
the tuning questions with **k = 3**. In the previous run they were scored at k = 6, a fifth of all 30
chunks, and nearly every setting got a perfect 1.0, so the grid could not tell them apart. With three
passages, a weaker setting actually loses facts. Ties are broken by the embedding model's average
over all settings (robustness), then by **lower redundancy**, which is what MMR is for.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=None)
def tokenizer_for(embed_key):
    # token-aware chunking must count tokens with the tokenizer of the model being tested
    return AutoTokenizer.from_pretrained(EMBEDDING_MODELS[embed_key]["model_name"])


def make_chunks(chunking, embed_key, add_headers=False):
    cfg = CHUNK_CONFIGS[chunking]
    tokenizer = tokenizer_for(embed_key) if cfg["method"] == "token" else None
    return split_documents(
        pages, add_headers=add_headers, embeddings=get_embeddings(embed_key), tokenizer=tokenizer, **cfg
    )


GRID_K = 3  # few passages, so weaker settings actually lose facts (k = 6 saturated at 1.0)

rows = []
for embed_key in EMBED_KEYS:
    embedder = get_embeddings(embed_key)
    for chunking in CHUNK_CONFIGS:
        for headers in (False, True):
            store = build_vectorstore(make_chunks(chunking, embed_key, headers), embedder, backend="faiss")
            for search_type in ("similarity", "mmr"):
                retriever = get_retriever(store, search_type, k=GRID_K, fetch_k=20, lambda_mult=0.75)
                retrieved = {i.question: retriever.invoke(i.question) for i in TUNING_SET if i.source}
                scores = evaluate_retrieval(retrieved.__getitem__, TUNING_SET)
                redundancy = np.mean([context_diversity(docs, embedder) for docs in retrieved.values()])
                rows.append(
                    {"embedding": embed_key, "chunking": chunking, "headers": headers, "search": search_type,
                     **scores, "redundancy": redundancy}
                )

grid = pd.DataFrame(rows)
grid["score"] = grid[["context_recall", "mrr", "top1_source_acc"]].mean(axis=1)
# A small question set makes the score coarse, so ties happen. They are broken by how well the embedding
# model does across *all* configurations (robustness), then by lower redundancy (less repetition).
grid["model_mean"] = grid.groupby("embedding").score.transform("mean")
grid = (
    grid.sort_values(["score", "model_mean", "context_recall", "redundancy"], ascending=[False, False, False, True])
    .reset_index(drop=True)
    .round(3)
)
display(grid.head(10))
print("Best score per embedding model / chunking (context recall):")
display(grid.pivot_table(index="chunking", columns="embedding", values="context_recall", aggfunc="max"))
display(grid.groupby("headers")[["context_recall", "mrr", "top1_source_acc"]].mean().round(3))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

,embedding,chunking,headers,search,context_recall,mrr,top1_source_acc,redundancy,score,model_mean
0,bge-base,recursive-800/0,False,mmr,1.000,1.000,1.0,0.703,1.000,0.918
1,bge-base,recursive-800/0,False,similarity,1.000,1.000,1.0,0.731,1.000,0.918
2,bge-base,recursive-800/0,True,mmr,1.000,0.941,1.0,0.752,0.980,0.918
3,bge-base,recursive-800/0,True,similarity,1.000,0.941,1.0,0.780,0.980,0.918
4,e5-base,recursive-800/0,False,similarity,1.000,0.941,1.0,0.854,0.980,0.917
5,e5-base,recursive-800/0,True,mmr,1.000,0.931,1.0,0.869,0.977,0.917
6,e5-base,recursive-800/0,True,similarity,1.000,0.931,1.0,0.885,0.977,0.917
7,e5-base,recursive-800/0,False,mmr,0.961,0.941,1.0,0.838,0.967,0.917
8,bge-base,recursive-1200/180,False,mmr,0.941,0.912,1.0,0.732,0.951,0.918
9,bge-base,recursive-1200/180,False,similarity,0.941,0.912,1.0,0.745,0.951,0.918


Best score per embedding model / chunking (context recall):


embedding,bge-base,bge-small,e5-base,minilm
chunking,,,,
character-800/120,0.941,0.941,0.882,1.000
recursive-1200/180,1.000,0.941,0.941,0.941
recursive-500/75,1.000,0.980,1.000,0.882
recursive-800/0,1.000,1.000,1.000,0.941
recursive-800/120,1.000,1.000,0.941,0.922
semantic-p90,0.941,1.000,0.941,0.941
token-160/32,0.941,1.000,0.941,0.941


,context_recall,mrr,top1_source_acc
headers,,,
False,0.928,0.824,0.962
True,0.934,0.837,0.964


In [ ]:
from functools import lru_cache
import numpy as np
import pandas as pd


@lru_cache(maxsize=None)
def tokenizer_for(embed_key):
    return AutoTokenizer.from_pretrained(EMBEDDING_MODELS[embed_key]["model_name"])


# Pre-generate chunks where possible
def get_or_make_chunks(chunking, embed_key, add_headers, embedder=None):
    cfg = CHUNK_CONFIGS[chunking]
    tokenizer = tokenizer_for(embed_key) if cfg["method"] == "token" else None
    return split_documents(
        pages, add_headers=add_headers, tokenizer=tokenizer, embeddings=embedder, **cfg
    )


GRID_K = 3
rows = []

for embed_key in EMBED_KEYS:
    embedder = get_embeddings(embed_key)

    # 1. Precompute query embeddings once per model
    eval_queries = [i.question for i in TUNING_SET if i.source]
    query_embeddings = {
        q: embedder.embed_query(q) for q in set(eval_queries)
    }

    for chunking in CHUNK_CONFIGS:
        for headers in (False, True):
            chunks = get_or_make_chunks(chunking, embed_key, headers, embedder)

            # 2. Build store once for both search strategies
            store = build_vectorstore(chunks, embedder, backend="faiss")

            for search_type in ("similarity", "mmr"):
                retriever = get_retriever(
                    store, search_type, k=GRID_K, fetch_k=20, lambda_mult=0.75
                )

                # Query using pre-warmed embeddings if supported by your vectorstore wrapper,
                # or retrieve normally
                retrieved = {
                    q: retriever.invoke(q) for q in eval_queries
                }

                scores = evaluate_retrieval(retrieved.__getitem__, TUNING_SET)
                redundancy = np.mean([
                    context_diversity(docs, embedder) for docs in retrieved.values()
                ])

                rows.append({
                    "embedding": embed_key,
                    "chunking": chunking,
                    "headers": headers,
                    "search": search_type,
                    **scores,
                    "redundancy": redundancy,
                })


In [ ]:
best = grid.iloc[0]
BEST = {"embedding": best.embedding, "chunking": best.chunking, "headers": bool(best.headers), "search": best.search}
print("Selected configuration:", BEST)

best_emb = get_embeddings(BEST["embedding"])
best_chunks = make_chunks(BEST["chunking"], BEST["embedding"], BEST["headers"])
vectorstore = build_vectorstore(
    best_chunks, best_emb, backend="chroma", persist_directory=CHROMA_DIR, collection_name="ast_best"
)
print(f"Final Chroma store: {vectorstore._collection.count()} chunks")

Selected configuration: {'embedding': 'bge-base', 'chunking': 'recursive-800/0', 'headers': False, 'search': 'mmr'}
Final Chroma store: 29 chunks


## 4. Retrieval-augmented generation with an open-source LLM

### 4.1 MMR, the grounded prompt and the answer self-check
**Similarity** search returns the k closest chunks, which often are near-duplicates (overlapping
chunks of the same paragraph). **MMR** (maximal marginal relevance) fetches `fetch_k` candidates and
greedily picks chunks that are relevant *and* unlike those already chosen; `lambda_mult` trades
relevance (1.0) against diversity (0.0).

The prompt numbers each passage with its document and page, tells the model to answer only from them,
to copy names character for character, to cite passage numbers, and to reply with a fixed sentence
when the answer isn't there.

Small models still slip, so every draft gets two cheap checks: is it **citation-only** (`"[1] [2][3]"`),
and does it name a **file that appears in neither the passages nor the question**? If either fails,
the model sees its draft plus the problem and rewrites once; the version with fewer problems is kept.
Names are flagged, never auto-corrected, because the nearest real name is often wrong: the
hallucinated `test_pipeline.py` is closest to `train_pipeline.py`, not to the correct `test_predictions.py`.

In [ ]:
NOT_FOUND = "I couldn't find that in the AST-1/AST-2 documents."

SYSTEM_PROMPT = f"""You are a teaching assistant for two course documents:
AST-1 "Transitioning from Research to Production Environment" and
AST-2 "Testing the Modules & Packaging".

Rules:
- Answer ONLY from the numbered context passages. Do not use outside knowledge.
- If the passages do not contain the answer, reply exactly: "{NOT_FOUND}"
- Write the answer in words first, then cite. Never reply with citation numbers alone.
- Be concise and specific. Use short bullet points for lists or steps.
- Copy every file name, folder name and command character for character from the passages.
  Never invent, merge or "correct" a name: if a passage says `report_builder.py`, write
  `report_builder.py`, not `report_pipeline.py`.
- Cite the passages you used with their numbers, e.g. [1] or [2][3]."""

# File-like names (scripts, configs, data, build artefacts) for the grounding check.
_FILENAME_RE = re.compile(
    r"(?<![\w.-])[\w-]+(?:\.[\w-]+)*\.(?:py|txt|toml|ini|in|cfg|yml|yaml|csv|json|ipynb|whl|gz|md)\b",
    re.IGNORECASE,
)
_CITATION_RE = re.compile(r"\[\d+\]")
_PASSAGE_HEADER_RE = re.compile(r"\[\d+\]\s*\((?:AST-\d|\?)")  # "[1] (AST-2, p.3)" copied from the context


def ungrounded_names(question: str, answer: str, docs: Sequence[Document]) -> list[str]:
    """File names in the answer that appear neither in the passages nor in the question.

    Names are *flagged*, never auto-corrected: the closest real name is often the wrong one
    (``test_pipeline.py`` is nearest to ``train_pipeline.py``, not ``test_predictions.py``).
    """
    known = (question + " " + " ".join(d.page_content for d in docs)).lower()
    names = dict.fromkeys(m.group() for m in _FILENAME_RE.finditer(answer))
    return [n for n in names if n.lower() not in known]


def answer_issues(question: str, answer: str, docs: Sequence[Document]) -> list[str]:
    """Cheap, deterministic checks on a generated answer (empty list = looks fine)."""
    issues = []
    words = re.findall(r"[A-Za-z]{2,}", _CITATION_RE.sub(" ", answer))
    if len(words) < 3:
        issues.append("it contains only citation numbers and no actual answer")
    if _PASSAGE_HEADER_RE.search(answer):
        issues.append("it copies the numbered passages instead of answering the question in its own words")
    missing = ungrounded_names(question, answer, docs)
    if missing:
        issues.append(
            "it mentions " + ", ".join(f"`{n}`" for n in missing) + ", which do not appear in the passages"
        )
    return issues


def format_context(docs: Sequence[Document]) -> str:
    """Numbered passages with document + page so the model can cite them."""
    blocks = []
    for i, d in enumerate(docs, 1):
        blocks.append(f"[{i}] ({d.metadata.get('doc_name', '?')}, {page_label(d)})\n{chunk_body(d)}")
    return "\n\n".join(blocks)


def build_messages(question: str, docs: Sequence[Document]) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{format_context(docs)}\n\nQuestion: {question}"},
    ]


def build_repair_messages(
    question: str, docs: Sequence[Document], draft: str, issues: Sequence[str]
) -> list[dict[str, str]]:
    """One self-correction turn: show the model its draft and what is wrong with it."""
    feedback = (
        "Your answer has a problem: "
        + "; ".join(issues)
        + ". Rewrite the full answer using only names and facts that appear verbatim in the "
        "passages above, then cite them."
    )
    return build_messages(question, docs) + [
        {"role": "assistant", "content": draft},
        {"role": "user", "content": feedback},
    ]

### 4.2 LLM backends
`local` runs an open-source instruct model with 🤗 transformers using greedy decoding, so answers
are reproducible. `groq`, `hf` and `openai` use any OpenAI-compatible endpoint. Groq and the Hugging
Face router serve open-weight models such as Llama 3.1. Both backends stream tokens.

In [ ]:
# 1.5B on GPU *and* CPU: in the Colab runs Qwen2.5-0.5B reached only 0.58 fact recall and
# produced citation-only replies ("[1] [2][3]"); 1.5B reached 0.75 with well-formed answers.
DEFAULT_LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


class LocalLLM:
    """Open-source chat model running locally with Hugging Face transformers."""

    def __init__(self, model: str | None = None, max_new_tokens: int = 512):
        import torch
        import transformers
        from transformers import AutoModelForCausalLM, AutoTokenizer

        cuda = torch.cuda.is_available()
        self.model_id = model or DEFAULT_LOCAL_MODEL
        self.max_new_tokens = max_new_tokens
        major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
        dtype_arg = "dtype" if (major, minor) >= (4, 56) else "torch_dtype"  # renamed in 4.56
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            **{dtype_arg: torch.float16 if cuda else torch.float32},
            device_map="auto" if cuda else None,
        )
        self.model.eval()

    def stream(self, messages: list[dict[str, str]]) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)
        errors: list[BaseException] = []
        tok = self.tokenizer
        pad_token_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id

        def generate():
            try:
                self.model.generate(
                    **inputs,
                    streamer=streamer,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,  # greedy = reproducible answers
                    temperature=None,
                    top_p=None,
                    top_k=None,
                    repetition_penalty=1.05,
                    pad_token_id=pad_token_id,
                )
            except BaseException as exc:  # e.g. CUDA OOM: unblock the consumer, re-raise below
                errors.append(exc)
                streamer.end()

        thread = threading.Thread(target=generate, daemon=True)
        thread.start()
        yield from streamer
        thread.join()
        if errors:
            raise RuntimeError(f"Generation failed: {errors[0]}") from errors[0]

    def invoke(self, messages: list[dict[str, str]]) -> str:
        return "".join(self.stream(messages)).strip()


# OpenAI-compatible endpoints serving open-weight models (plus OpenAI itself).
API_PRESETS: dict[str, dict[str, str | None]] = {
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "key_env": "GROQ_API_KEY",
        "model": "llama-3.1-8b-instant",
    },
    "hf": {
        "base_url": "https://router.huggingface.co/v1",
        "key_env": "HF_TOKEN",
        "model": "meta-llama/Llama-3.1-8B-Instruct",
    },
    "openai": {"base_url": None, "key_env": "OPENAI_API_KEY", "model": "gpt-4o-mini"},
}


class APIChatLLM:
    """Any OpenAI-compatible chat endpoint (Groq, Hugging Face router, OpenAI, ...)."""

    def __init__(
        self,
        provider: str = "groq",
        model: str | None = None,
        api_key: str | None = None,
        max_tokens: int = 512,
    ):
        from openai import OpenAI

        preset = API_PRESETS[provider]
        key = api_key or os.environ.get(str(preset["key_env"]))
        if not key:
            raise ValueError(f"Set the {preset['key_env']} environment variable to use '{provider}'.")
        self.client = OpenAI(base_url=preset["base_url"], api_key=key)
        self.model_id = model or os.environ.get("LLM_MODEL") or str(preset["model"])
        self.max_tokens = max_tokens

    def stream(self, messages: list[dict[str, str]]) -> Iterator[str]:
        response = self.client.chat.completions.create(
            model=self.model_id, messages=messages, temperature=0.0, max_tokens=self.max_tokens, stream=True
        )
        for chunk in response:
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content

    def invoke(self, messages: list[dict[str, str]]) -> str:
        return "".join(self.stream(messages)).strip()


def resolve_backend(backend: str = "auto") -> str:
    """ "auto" -> the first API provider whose key is set, otherwise "local"."""
    if backend != "auto":
        return backend
    return next((p for p in API_PRESETS if os.environ.get(str(API_PRESETS[p]["key_env"]))), "local")


def load_llm(backend: str = "local", model: str | None = None, **kwargs):
    """backend: "local" | "groq" | "hf" | "openai" | "auto". ``model`` overrides the default id."""
    backend = resolve_backend(backend)
    if backend == "local":
        return LocalLLM(model=model, **kwargs)
    return APIChatLLM(provider=backend, model=model, **kwargs)

In [ ]:
llm = load_llm(LLM_BACKEND, model=LLM_MODEL)
print("LLM:", llm.model_id)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM: Qwen/Qwen2.5-1.5B-Instruct


### 4.3 The pipeline (plain Python for streaming, plus the same thing as a LangChain LCEL chain)

In [ ]:
@dataclass
class RAGResult:
    question: str
    answer: str
    docs: list[Document] = field(default_factory=list)
    issues: list[str] = field(default_factory=list)  # problems left after the self-check
    repaired: bool = False  # True when the self-check rewrote the first draft


@dataclass
class RAGPipeline:
    """Retrieve -> grounded prompt -> LLM -> self-check. Settings can be overridden per call.

    ``self_check``: if the draft is citation-only or names a file that is not in the
    passages, the model gets one chance to rewrite it; the version with fewer problems wins.
    """

    vectorstore: object
    llm: object
    search_type: str = "mmr"
    k: int = 6
    fetch_k: int = 20
    lambda_mult: float = 0.75
    self_check: bool = True

    def _settings(self, **overrides) -> dict:
        settings = dict(
            search_type=self.search_type, k=self.k, fetch_k=self.fetch_k, lambda_mult=self.lambda_mult
        )
        settings.update({k: v for k, v in overrides.items() if v is not None})
        return settings

    def retrieve(self, question: str, **overrides) -> list[Document]:
        return get_retriever(self.vectorstore, **self._settings(**overrides)).invoke(question)

    def generate(self, question: str, docs: list[Document]) -> RAGResult:
        draft = self.llm.invoke(build_messages(question, docs))
        issues = answer_issues(question, draft, docs)
        if not (issues and self.self_check):
            return RAGResult(question, draft, docs, issues)
        fixed = self.llm.invoke(build_repair_messages(question, docs, draft, issues)).strip()
        fixed_issues = answer_issues(question, fixed, docs)
        if len(fixed_issues) < len(issues):
            return RAGResult(question, fixed, docs, fixed_issues, repaired=True)
        return RAGResult(question, draft, docs, issues)

    def answer(self, question: str, **overrides) -> RAGResult:
        return self.generate(question, self.retrieve(question, **overrides))

    def stream(self, question: str, **overrides) -> Iterator[tuple[str, list[Document]]]:
        """Yields (answer_so_far, docs) for the Gradio app, including the self-check rewrite."""
        docs = self.retrieve(question, **overrides)
        draft = ""
        yield draft, docs
        for token in self.llm.stream(build_messages(question, docs)):
            draft += token
            yield draft, docs
        draft = draft.strip()
        issues = answer_issues(question, draft, docs)
        if not (issues and self.self_check):
            yield draft, docs
            return
        yield draft + "\n\n_Checking the answer against the sources…_", docs
        fixed = ""
        for token in self.llm.stream(build_repair_messages(question, docs, draft, issues)):
            fixed += token
            yield fixed, docs
        fixed = fixed.strip()
        better = len(answer_issues(question, fixed, docs)) < len(issues)
        yield (fixed if better else draft), docs

    def as_chain(self):
        """The same pipeline as a LangChain LCEL runnable: question -> {answer, docs}."""
        from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

        retriever = get_retriever(self.vectorstore, **self._settings())
        generate = RunnableLambda(lambda x: self.generate(x["question"], x["docs"]).answer)
        return RunnableParallel(docs=retriever, question=RunnablePassthrough()) | RunnableParallel(
            answer=generate, docs=RunnableLambda(lambda x: x["docs"])
        )


_MD_SPECIAL = re.compile(r"([\\`*_{}\[\]<>()#+\-.!|~])")


_LIST_START = re.compile(r"^(\d+[.)]|[A-Z][.)]|[●•▪◦–-]|Step\s*\d)")


def _reflow(text: str) -> str:
    """Undo the PDF's hard line wraps, keeping breaks before list items and bullets."""
    lines: list[str] = []
    for line in (raw.strip() for raw in text.splitlines()):
        if line and lines and lines[-1] and not _LIST_START.match(line):
            lines[-1] += " " + line
        else:
            lines.append(line)
    return "\n".join(lines)


def _quote_literally(text: str) -> str:
    """Markdown blockquote that shows PDF text as-is (no accidental lists, bold, headings)."""
    lines = [_MD_SPECIAL.sub(r"\\\1", line) for line in _reflow(text).splitlines()]
    return "\n".join(f"> {line}  " if line.strip() else ">" for line in lines)


def sources_markdown(docs: Sequence[Document], max_chars: int = 400) -> str:
    """Retrieved passages formatted for display (Gradio / notebook)."""
    blocks = []
    for i, d in enumerate(docs, 1):
        body = chunk_body(d)
        snippet = body[:max_chars] + ("…" if len(body) > max_chars else "")
        blocks.append(
            f"**[{i}] {d.metadata.get('doc_name')}, {page_label(d)}**\n\n{_quote_literally(snippet)}"
        )
    return "\n\n".join(blocks)

In [ ]:
pipeline = RAGPipeline(vectorstore, llm, search_type=BEST["search"], k=6, fetch_k=20, lambda_mult=0.75)

chain = pipeline.as_chain()  # LCEL: question -> {"answer", "docs"}
result = chain.invoke("What is the role of conftest.py?")
display(Markdown(result["answer"]))
display(Markdown(sources_markdown(result["docs"], max_chars=250)))

The role of conftest.py is to define fixtures for pytest and sample_input_data function, which is passed to each test case. The sample_input_data function returns the test set with the target used for feature testing and prediction. [1]

**[1] AST-2, p.3**

> 1\. conftest\.py: Here we are defining a fixture for pytest and then sample\_input\_data function is passed to each test case\. The sample\_input\_data function returns the test set with the target which is used further for feature testing and prediction\. 2…  

**[2] AST-2, p.3**

> The key lines to do with packaging are one to six, where we specify what the basic dependencies for installing the packages are i\.e\. setuptools and wheel\. We might see some pyproject\.toml lines which specify alternative tools for building\. These are …  

**[3] AST-2, p.4**

> The Manifest\.in file is used to specify the files that should be included or excluded in the package\. This file is responsible for ensuring that important files like the pickle file, as well as the train and test CSVs, are included in the package\. Th…  

**[4] AST-1, p.3**

> 3\. data\_manger\.py: It contains pre\-pipeline functions, and functionality for loading of data, saving & loading of trained models, and removing old version models\.  
> 4\. features\.py : It contains custom built transformation classes if any\.  
> 5\. validation\.…  

**[5] AST-1, pp.3-4**

> ● \_\_init\_\_\.py : This file lets the Python interpreter know that a directory contains code for a Python module\. An \_\_init\_\_\.py file can be blank\. Without this, importing modules from another folder into your project is not possible\.  
>
> \[Screenshot text …  

**[6] AST-1, p.1**

> 1\. VS code setup and understanding interface components  
> 2\. Creating virtual environment, installing requirement, running \.py & \.ipynb file  
> 3\. Understanding \_\_name\_\_ == “\_\_main\_\_” and building our own modules  
> 4\. Modularization: Converting a ML model f…  

### 4.4 The self-check on the question 0.5B and 1.5B models got wrong

In [ ]:
# Question 10 of the held-out set (§5): small models answered "test_pipeline.py" here
q10 = "Where was the test-set prediction and accuracy calculation moved from train_pipeline.py?"
docs = pipeline.retrieve(q10)
for check in (False, True):
    res = replace(pipeline, self_check=check).generate(q10, docs)
    status = "repaired" if res.repaired else ("issues: " + "; ".join(res.issues) if res.issues else "no issues")
    display(Markdown(f"**self_check={check}** ({status})\n\n{res.answer}\n\n---"))

**self_check=False** (no issues)

In the provided procedure, the test-set prediction and accuracy calculation were moved to the `test_predictions.py` file.

---

**self_check=True** (no issues)

In the provided procedure, the test-set prediction and accuracy calculation were moved to the `test_predictions.py` file.

---

### 4.5 With vs without MMR

In [ ]:
COMPARE_QUESTIONS = [
    "How do I set up the project and run it in VS Code?",  # procedure described in both guides
    "Which tools are used for code styling, linting, type checking and import sorting?",
    "Which files and folders make up the production project structure?",
]

rows = []
for question in COMPARE_QUESTIONS:
    for search_type in ("similarity", "mmr"):
        res = pipeline.answer(question, search_type=search_type, k=6, lambda_mult=0.75)
        rows.append(
            {
                "question": question,
                "search": search_type,
                "retrieved": ", ".join(f"{d.metadata['doc_name']} {page_label(d)}" for d in res.docs),
                "distinct pages": len({(d.metadata["doc_name"], d.metadata["page"]) for d in res.docs}),
                "redundancy (mean pairwise cosine)": context_diversity(res.docs, best_emb),
                "answer": res.answer,
            }
        )
mmr_vs_sim = pd.DataFrame(rows)
display(mmr_vs_sim.drop(columns="answer").round(3))
for r in rows:
    display(Markdown(f"**{r['search']}** · *{r['question']}*\n\n{r['answer']}\n\n---"))

,question,search,retrieved,distinct pages,redundancy (mean pairwise cosine)
0,How do I set up the project and run it in VS Code?,similarity,"AST-1 p.4, AST-2 p.5, AST-1 p.4, AST-1 p.1, AST-1 pp.4-5, AST-2 pp.3-4",4,0.705
1,How do I set up the project and run it in VS Code?,mmr,"AST-1 p.4, AST-2 p.5, AST-1 p.4, AST-1 p.1, AST-1 pp.4-5, AST-2 pp.3-4",4,0.705
2,"Which tools are used for code styling, linting, type checking and import sorting?",similarity,"AST-2 pp.4-5, AST-2 p.3, AST-2 p.2, AST-2 p.3, AST-1 p.1, AST-1 p.1",4,0.671
3,"Which tools are used for code styling, linting, type checking and import sorting?",mmr,"AST-2 pp.4-5, AST-2 p.3, AST-2 p.2, AST-1 p.1, AST-1 p.1, AST-1 p.3",5,0.651
4,Which files and folders make up the production project structure?,similarity,"AST-1 p.2, AST-1 pp.2-3, AST-1 p.1, AST-2 p.5, AST-1 p.4, AST-2 pp.2-3",5,0.649
5,Which files and folders make up the production project structure?,mmr,"AST-1 p.2, AST-1 pp.2-3, AST-1 p.1, AST-2 p.5, AST-1 p.4, AST-2 pp.2-3",5,0.649


**similarity** · *How do I set up the project and run it in VS Code?*

To set up the project and run it in VS Code, follow these steps:

1. **Environment Setup**:
   - Go through the Environment setup document.
   - Complete the VS code installation and Python setup on your local system.

2. **Upload Project to VS Code**:
   - Upload the `project_with_test_and_package` folder in VS Code.
   - Create a virtual environment within the project.
   - Install necessary dependencies.
   - Execute specific scripts for training and prediction.

3. **Run Scripts**:
   - Upload the project folder in VS Code.
   - Create a virtual environment:
     - Open terminal.
     - Navigate to the project directory.
     - Run the following command to create the virtual environment:
       ```bash
       python -m venv path_of_project_folder/venv
       ```
     - Example:
       ```bash
       C:\Users\karna\Desktop\MLOps_M3_M4\project> python -m venv C:\Users\karna\Desktop\MLOps_M3_M4\project\venv
       ```

4. **Activate Virtual Environment**:
   - After creating the virtual environment, activate it:
     ```bash
     (venv) C:\Users\karna\Desktop\MLOps_M3_M4\project>
     ```

5. **Run Scripts**:
   - Run the following commands to execute the necessary scripts:
     ```bash
     pip install -r requirements.txt
     python train_pipeline.py
     python predict.py
     ```

This setup allows you to effectively configure the project and execute it within the VS Code environment.

---

**mmr** · *How do I set up the project and run it in VS Code?*

To set up the project and run it in VS Code, follow these steps:

1. **Environment Setup**:
   - Go through the Environment setup document.
   - Complete the VS code installation and Python setup on your local system.

2. **Upload Project to VS Code**:
   - Upload the `project_with_test_and_package` folder in VS Code.
   - Create a virtual environment within the project.
   - Install necessary dependencies.
   - Execute specific scripts for training and prediction.

3. **Run Scripts**:
   - Upload the project folder in VS Code.
   - Create a virtual environment:
     - Open terminal.
     - Navigate to the project directory.
     - Run the following command to create the virtual environment:
       ```bash
       python -m venv path_of_project_folder/venv
       ```
     - Example:
       ```bash
       C:\Users\karna\Desktop\MLOps_M3_M4\project> python -m venv C:\Users\karna\Desktop\MLOps_M3_M4\project\venv
       ```

4. **Activate Virtual Environment**:
   - After creating the virtual environment, activate it:
     ```bash
     (venv) C:\Users\karna\Desktop\MLOps_M3_M4\project>
     ```

5. **Run Scripts**:
   - Run the following commands to execute the necessary scripts:
     ```bash
     pip install -r requirements.txt
     python train_pipeline.py
     python predict.py
     ```

This setup allows you to effectively configure the project and execute it within the VS Code environment.

---

**similarity** · *Which tools are used for code styling, linting, type checking and import sorting?*

I couldn't find that in the AST-1/AST-2 documents.

---

**mmr** · *Which tools are used for code styling, linting, type checking and import sorting?*

I couldn't find that in the AST-1/AST-2 documents.

---

**similarity** · *Which files and folders make up the production project structure?*

project
requirements
requirements.txt
titanic_model
config
datasets
processing
trained_models
config.yml
pipeline.py
predict.py
train_pipeline.py
VERSION
_init_.py

project
requirements
titanic_model

Files and folders that make up the production project structure include:

1. **requirements**: Contains dependencies and libraries.
2. **core.py**: Accesses the `config.yml` file and sets paths for dataset, trained models, etc.
3. **titanic_model**: Contains the main model and its configurations.
4. **config.yml**: Configuration file for the model.
5. **pipeline.py**: Handles the overall pipeline logic.
6. **predict.py**: Contains prediction logic.
7. **train_pipeline.py**: Handles the training pipeline.
8. **VERSION**: Version control information.
9. **_init_.py**: Initializes the project.

Files and folders for testing include:

1. **requirements.txt**: List of required packages for testing.
2. **config.yml**: Configuration file for testing.
3. **pipeline.py**: Contains test cases for the pipeline.
4. **predict.py**: Contains test cases for prediction logic.

---

**mmr** · *Which files and folders make up the production project structure?*

project
requirements
requirements.txt
titanic_model
config
datasets
processing
trained_models
config.yml
pipeline.py
predict.py
train_pipeline.py
VERSION
_init_.py

project
requirements
titanic_model

Files and folders that make up the production project structure include:

1. **requirements**: Contains dependencies and libraries.
2. **core.py**: Accesses the `config.yml` file and sets paths for dataset, trained models, etc.
3. **titanic_model**: Contains the main model and its configurations.
4. **config.yml**: Configuration file for the model.
5. **pipeline.py**: Handles the overall pipeline logic.
6. **predict.py**: Contains prediction logic.
7. **train_pipeline.py**: Handles the training pipeline.
8. **VERSION**: Version control information.
9. **_init_.py**: Initializes the project.

Files and folders for testing include:

1. **requirements.txt**: List of required packages for testing.
2. **config.yml**: Configuration file for testing.
3. **pipeline.py**: Contains test cases for the pipeline.
4. **predict.py**: Contains test cases for prediction logic.

---

In [ ]:
# k × λ sweep on the tuning questions (retrieval only, no LLM calls)
def redundancy_of(retriever):
    return np.mean([context_diversity(retriever.invoke(i.question), best_emb) for i in TUNING_SET if i.source])


sweep = []
for k in (2, 4, 6):
    for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
        retriever = get_retriever(vectorstore, "mmr", k=k, fetch_k=20, lambda_mult=lam)
        scores = evaluate_retrieval(retriever.invoke, TUNING_SET)
        sweep.append({"k": k, "retriever": f"mmr λ={lam}", **scores, "redundancy": redundancy_of(retriever)})
    retriever = get_retriever(vectorstore, "similarity", k=k)
    scores = evaluate_retrieval(retriever.invoke, TUNING_SET)
    sweep.append({"k": k, "retriever": "similarity", **scores, "redundancy": redundancy_of(retriever)})
lambda_sweep = pd.DataFrame(sweep).round(3)
display(lambda_sweep)

,k,retriever,context_recall,mrr,top1_source_acc,redundancy
0,2,mmr λ=0.0,0.961,1.0,1.0,0.535
1,2,mmr λ=0.25,0.961,1.0,1.0,0.539
2,2,mmr λ=0.5,0.961,1.0,1.0,0.572
3,2,mmr λ=0.75,0.961,1.0,1.0,0.694
4,2,mmr λ=1.0,1.000,1.0,1.0,0.743
5,2,similarity,1.000,1.0,1.0,0.743
6,4,mmr λ=0.0,0.961,1.0,1.0,0.548
7,4,mmr λ=0.25,0.961,1.0,1.0,0.547
8,4,mmr λ=0.5,0.961,1.0,1.0,0.592
9,4,mmr λ=0.75,1.000,1.0,1.0,0.672


Lower *redundancy* means the k passages repeat each other less, leaving more room for distinct facts.
λ = 1.0 behaves like similarity search. Read the **k = 2** rows first: with only two passages, a
"diverse" but off-topic pick costs a needed fact, so that is where too low a λ shows up as lower recall.
At k = 6, recall tends to saturate on these short guides (the previous run scored 1.00 at every λ), and
MMR's effect shows only as lower redundancy (0.70 for similarity → 0.58 at λ = 0 in that run).
The default k = 6, λ = 0.75 keeps full recall with less repetition than similarity search.

## 5. Questions and answers

The held-out evaluation set has ten questions spanning both guides: file roles, the setup procedure,
testing theory, packaging files, tooling and build commands. It adds one out-of-scope question to check
that the system declines instead of inventing an answer. None of these questions were used for tuning.

In [ ]:
EVAL_SET: list[EvalItem] = [
    EvalItem(
        "What is the purpose of data_manager.py in the titanic_model project?",
        [
            ["pre-pipeline", "pre pipeline"],
            [
                "loading of data",
                "load data",
                "loading data",
                "loads data",
                "load the data",
                "loading the data",
            ],
            ["old version", "old model", "older version", "previous version"],
        ],
        "AST-1",
    ),
    EvalItem(
        "How do I create and activate a virtual environment for the project in VS Code?",
        [["python -m venv"], ["activate"], ["interpreter"]],
        "AST-1",
    ),
    EvalItem(
        "What makes production code different from research code?",
        [["end users", "end-users"], ["experiment", "proof of concept", "proofs of concept"], ["maintain"]],
        "AST-1",
    ),
    EvalItem(
        "What do the VERSION file and the __init__.py file do?",
        [["version tag"], ["python module", "python package", "module"]],
        "AST-1",
    ),
    EvalItem(
        "What are the three types of tests and how do they differ?",
        [["unit test"], ["integration test"], ["system test"]],
        "AST-2",
    ),
    EvalItem(
        "Which files are related to packaging the model?",
        [["pyproject.toml"], ["setup.py"], ["manifest.in"], ["mypy.ini", "mypy.in"]],
        "AST-2",
    ),
    EvalItem(
        "What is the role of conftest.py?",
        [["fixture"], ["sample_input_data", "sample input data"]],
        "AST-2",
    ),
    EvalItem(
        "Which tools are used for code styling, linting, type checking and import sorting?",
        [["black"], ["flake8"], ["mypy"], ["isort"]],
        "AST-2",
    ),
    EvalItem(
        "What commands build the package and which folders are created afterwards?",
        [["-m build"], ["dist"], ["egg-info", "egg info"]],
        "AST-2",
    ),
    EvalItem(
        "Where was the test-set prediction and accuracy calculation moved from train_pipeline.py?",
        [["test_predictions.py", "test_prediction.py"]],
        "AST-2",
    ),
    EvalItem("What is the capital of France?", [], None),
]


def evaluate_answers(
    pipeline: RAGPipeline, eval_set: Sequence[EvalItem] = EVAL_SET, **overrides
) -> list[dict]:
    """Run every question through the full RAG pipeline and score the answers."""
    rows = []
    for item in eval_set:
        result = pipeline.answer(item.question, **overrides)
        in_scope = item.source is not None
        rows.append(
            {
                "question": item.question,
                "answer": result.answer,
                "fact_recall": fact_recall(result.answer, item.facts) if in_scope else None,
                "correct_refusal": (not is_refusal(result.answer)) if in_scope else is_refusal(result.answer),
                "repaired": result.repaired,
                "sources": ", ".join(f"{d.metadata['doc_name']} {page_label(d)}" for d in result.docs),
            }
        )
    return rows

In [ ]:
display(
    pd.DataFrame(
        [{"question": item.question, "expected source": item.source or "out of scope"} for item in EVAL_SET],
        index=range(1, len(EVAL_SET) + 1),
    )
)

,question,expected source
1,What is the purpose of data_manager.py in the titanic_model project?,AST-1
2,How do I create and activate a virtual environment for the project in VS Code?,AST-1
3,What makes production code different from research code?,AST-1
4,What do the VERSION file and the __init__.py file do?,AST-1
5,What are the three types of tests and how do they differ?,AST-2
6,Which files are related to packaging the model?,AST-2
7,What is the role of conftest.py?,AST-2
8,"Which tools are used for code styling, linting, type checking and import sorting?",AST-2
9,What commands build the package and which folders are created afterwards?,AST-2
10,Where was the test-set prediction and accuracy calculation moved from train_pipeline.py?,AST-2


In [ ]:
answers = pd.DataFrame(evaluate_answers(pipeline, EVAL_SET))
for i, row in answers.iterrows():
    display(Markdown(f"**Q{i + 1}. {row.question}**\n\n{row.answer}\n\n<sub>Sources: {row.sources}</sub>\n\n---"))
display(answers[["question", "fact_recall", "correct_refusal", "repaired"]])
print(f"Mean fact recall (in-scope): {answers.fact_recall.mean():.2f}")
print(f"Correct behaviour (answered in-scope, declined out-of-scope): {answers.correct_refusal.mean():.2f}")
print(f"Answers rewritten by the self-check: {int(answers.repaired.sum())} of {len(answers)}")

**Q1. What is the purpose of data_manager.py in the titanic_model project?**

The purpose of data_manager.py in the titanic_model project is to contain pre-pipeline functions and functionality for loading, saving, and loading of trained models, as well as removing old versions of models.

<sub>Sources: AST-1 pp.2-3, AST-2 p.4, AST-2 p.6, AST-1 pp.3-4, AST-2 pp.3-4, AST-1 p.3</sub>

---

**Q2. How do I create and activate a virtual environment for the project in VS Code?**

To create and activate a virtual environment for the project in VS Code, follow these steps:

1. **Open Terminal**: Navigate to the root directory of your project using the command prompt or terminal.

2. **Activate Virtual Environment**:
   - Type the following command in the terminal:
     ```bash
     project\venv\Scripts> activate
     ```
   - This command activates the virtual environment located at `C:\Users\karna\Desktop\MLOps_M3_M4\project\venv`.

3. **Verify Activation**:
   - You can verify that the virtual environment is activated by checking the terminal prompt:
     ```bash
     (venv) ...
     ```

4. **Deactivate Virtual Environment**:
   - To deactivate the virtual environment, simply type:
     ```bash
     deactivate
     ```

By following these steps, you will successfully create and activate a virtual environment for your project in VS Code.

<sub>Sources: AST-1 p.4, AST-2 p.5, AST-1 p.4, AST-1 p.1, AST-1 pp.4-5, AST-2 pp.3-4</sub>

---

**Q3. What makes production code different from research code?**

Production code is designed to be deployed to end users as opposed to research code, which is for experimentation, building proof of concepts, and research code tends to be more short-term in nature.

<sub>Sources: AST-1 p.1, AST-1 p.1, AST-1 p.2, AST-2 p.2, AST-1 p.1, AST-2 p.6</sub>

---

**Q4. What do the VERSION file and the __init__.py file do?**

The VERSION file contains the version tag, which is crucial for identifying the current state of the project. The __init__.py file lets the Python interpreter know that a directory contains code for a Python module, allowing imports from other folders within the project.

<sub>Sources: AST-1 pp.3-4, AST-2 p.4, AST-2 pp.3-4, AST-2 p.4, AST-1 p.3, AST-1 p.1</sub>

---

**Q5. What are the three types of tests and how do they differ?**

Three types of tests are designed: Unit test, Integration test, and system test. 

A unit test is the smallest and simplest form of software testing, focusing on assessing a separable unit of software like a class or a function for correctness, independent of the larger software system composed of many such units. 

An integration test checks the interaction between different modules or components of a software system to ensure they work together correctly. 

A system test covers the entire system, verifying that all components function as expected when combined.

<sub>Sources: AST-2 p.2, AST-2 p.3, AST-2 p.1, AST-2 p.2, AST-2 p.2, AST-2 p.1</sub>

---

...

**Q9. What commands build the package and which folders are created afterwards?**

To build the package, run the following commands:

1. Run: `py -m pip install --upgrade build`
2. Run: `py -m build`

After running these commands, you will see the following result:

You will notice, two additional folders will be created:
1. dist: This contains the distributable 'whl' file and 'gz tar' file. The 'whl' file is easy to install.
2. titanic_model.egg-info: This contains metadata and information of the packages along with the requirements information.

These folders are:
- **dist**: Contains the distributable files like `.whl` and `.tar.gz`.
- **titanic_model.egg-info**: Contains metadata and information about the package, including its requirements.

<sub>Sources: AST-2 p.4, AST-2 pp.5-6, AST-2 p.3, AST-2 p.5, AST-2 pp.2-3, AST-2 p.1</sub>

---

**Q10. Where was the test-set prediction and accuracy calculation moved from train_pipeline.py?**

In the provided procedure, the test-set prediction and accuracy calculation were moved to the `test_predictions.py` file.

<sub>Sources: AST-1 p.3, AST-2 p.5, AST-2 p.3, AST-2 pp.5-6, AST-2 pp.4-5, AST-1 pp.2-3</sub>

---

**Q11. What is the capital of France?**

I couldn't find that in the AST-1/AST-2 documents.

<sub>Sources: AST-1 p.3, AST-1 p.4, AST-2 p.3, AST-1 p.3, AST-1 pp.4-5, AST-1 p.1</sub>

---

,question,fact_recall,correct_refusal,repaired
0,What is the purpose of data_manager.py in the titanic_model project?,0.666667,True,False
1,How do I create and activate a virtual environment for the project in VS Code?,0.333333,True,False
2,What makes production code different from research code?,0.666667,True,False
3,What do the VERSION file and the __init__.py file do?,1.000000,True,False
4,What are the three types of tests and how do they differ?,1.000000,True,False
5,Which files are related to packaging the model?,1.000000,True,False
6,What is the role of conftest.py?,1.000000,True,False
7,"Which tools are used for code styling, linting, type checking and import sorting?",0.000000,False,False
8,What commands build the package and which folders are created afterwards?,1.000000,True,False
9,Where was the test-set prediction and accuracy calculation moved from train_pipeline.py?,1.000000,True,False


Mean fact recall (in-scope): 0.77
Correct behaviour (answered in-scope, declined out-of-scope): 0.91
Answers rewritten by the self-check: 0 of 11


### 5.1 Answer-level hyperparameter sweep (tuning questions)

In [ ]:
VARIANTS = {
    # λ <= 0.25 is left out: it lowered recall in the retrieval sweep (§4.5)
    "current": dict(search_type=pipeline.search_type, k=pipeline.k, lambda_mult=pipeline.lambda_mult),
    "k=4, mmr λ=0.75": dict(search_type="mmr", k=4, lambda_mult=0.75),
    "k=6, mmr λ=0.5": dict(search_type="mmr", k=6, lambda_mult=0.5),
    "k=6, mmr λ=0.75": dict(search_type="mmr", k=6, lambda_mult=0.75),
    "k=8, mmr λ=0.75": dict(search_type="mmr", k=8, lambda_mult=0.75),
    "k=6, similarity": dict(search_type="similarity", k=6),
}
if RUN_ANSWER_GRID:
    results = []
    for name, settings in VARIANTS.items():
        scored = pd.DataFrame(evaluate_answers(pipeline, TUNING_SET, **settings))
        results.append(
            {
                "variant": name,
                "answer fact recall": scored.fact_recall.mean(),
                "correct behaviour": scored.correct_refusal.mean(),
            }
        )
    answer_grid = pd.DataFrame(results)
    answer_grid["total"] = answer_grid["answer fact recall"] + answer_grid["correct behaviour"]
    # stable sort keeps "current" first on ties: only switch settings for a strict improvement
    answer_grid = answer_grid.sort_values("total", ascending=False, kind="stable").reset_index(drop=True)
    display(answer_grid.round(3))
    pipeline = replace(pipeline, **VARIANTS[answer_grid.variant.iloc[0]])
print("Pipeline settings:", {k: getattr(pipeline, k) for k in ("search_type", "k", "fetch_k", "lambda_mult")})

,variant,answer fact recall,correct behaviour,total
0,"k=8, mmr λ=0.75",1.000,0.944,1.944
1,"k=6, mmr λ=0.5",0.902,0.944,1.846
2,"k=6, similarity",0.941,0.889,1.830
3,current,0.882,0.889,1.771
4,"k=6, mmr λ=0.75",0.882,0.889,1.771
5,"k=4, mmr λ=0.75",0.824,0.889,1.712


Pipeline settings: {'search_type': 'mmr', 'k': 8, 'fetch_k': 20, 'lambda_mult': 0.75}


### 5.2 A question only the screenshots can answer

In [ ]:
image_question = "How many tests were collected and how many passed in the pytest run shown in AST-2?"
res = pipeline.answer(image_question)
in_context = any("2 passed" in d.page_content for d in res.docs)
display(Markdown(f"**{image_question}**\n\n{res.answer}\n\n<sub>OCR text retrieved: {in_context}</sub>"))

**How many tests were collected and how many passed in the pytest run shown in AST-2?**

2 tests were collected, and both passed.

<sub>OCR text retrieved: True</sub>

In [ ]:
by_model = grid.groupby("embedding").score.max().sort_values(ascending=False)
by_search = grid.groupby("search")[["context_recall", "mrr"]].mean()
findings = [
    f"- **Retrieval configuration** (tuning set): `{BEST['embedding']}` embeddings, `{BEST['chunking']}` chunks, "
    f"headers {'on' if BEST['headers'] else 'off'}, `{BEST['search']}` search, score {grid.score.iloc[0]:.2f}.",
    "- **Embedding models** (best score each): " + ", ".join(f"`{k}` {v:.2f}" for k, v in by_model.items()) + ".",
    f"- **Contextual headers**: mean top-1 source accuracy {grid[grid.headers].top1_source_acc.mean():.2f} with, "
    f"{grid[~grid.headers].top1_source_acc.mean():.2f} without.",
    f"- **Chunk overlap** (chosen setting): {sum(o > 0 for o in overlaps_of(best_chunks))} of "
    f"{len(overlaps_of(best_chunks))} boundaries overlap, by {min(overlaps_of(best_chunks))}–"
    f"{max(overlaps_of(best_chunks))} characters.",
    f"- **Similarity vs MMR** (all configs, k = {GRID_K}): context recall {by_search.context_recall['similarity']:.2f} vs "
    f"{by_search.context_recall['mmr']:.2f}.",
    "- **k × λ sweep**: "
    + "; ".join(
        f"k={r.k} {r.retriever}: recall {r.context_recall:.2f}, redundancy {r.redundancy:.2f}"
        for r in lambda_sweep.itertuples()
        if r.retriever in ("mmr λ=0.25", "mmr λ=0.75", "similarity")
    )
    + ".",
    f"- **Held-out answers**: mean fact recall {answers.fact_recall.mean():.2f}; correct behaviour on "
    f"{answers.correct_refusal.mean():.0%} of questions, including the out-of-scope probe; "
    f"{int(answers.repaired.sum())} answer(s) rewritten by the self-check.",
    f"- **OCR**: {'on' if OCR_IMAGES else 'off'}; screenshot fact retrieved for the image-only question: {in_context}.",
]
display(Markdown("### Findings (generated from this run)\n" + "\n".join(findings)))

### Findings (generated from this run)
- **Retrieval configuration** (tuning set): `bge-base` embeddings, `recursive-800/0` chunks, headers off, `mmr` search, score 1.00.
- **Embedding models** (best score each): `bge-base` 1.00, `e5-base` 0.98, `bge-small` 0.95, `minilm` 0.95.
- **Contextual headers**: mean top-1 source accuracy 0.96 with, 0.96 without.
- **Chunk overlap** (chosen setting): 0 of 27 boundaries overlap, by -2–-1 characters.
- **Similarity vs MMR** (all configs, k = 3): context recall 0.93 vs 0.93.
- **k × λ sweep**: k=2 mmr λ=0.25: recall 0.96, redundancy 0.54; k=2 mmr λ=0.75: recall 0.96, redundancy 0.69; k=2 similarity: recall 1.00, redundancy 0.74; k=4 mmr λ=0.25: recall 0.96, redundancy 0.55; k=4 mmr λ=0.75: recall 1.00, redundancy 0.67; k=4 similarity: recall 1.00, redundancy 0.72; k=6 mmr λ=0.25: recall 0.96, redundancy 0.57; k=6 mmr λ=0.75: recall 1.00, redundancy 0.67; k=6 similarity: recall 1.00, redundancy 0.70.
- **Held-out answers**: mean fact recall 0.77; correct behaviour on 91% of questions, including the out-of-scope probe; 0 answer(s) rewritten by the self-check.
- **OCR**: on; screenshot fact retrieved for the image-only question: True.

## 6. Gradio app

The answer streams in as it is generated. The passages it was grounded on are shown beside it with
document and page, and the retrieval settings (MMR or similarity, k, λ) can be changed live. If the
self-check rewrites an answer you see it happen, and any file name that still can't be found in the
sources is flagged under the answer. In Colab, `share=True` prints a public link.

In [ ]:
import gradio as gr

EXAMPLES = [
    "What is the purpose of data_manager.py?",
    "How do I create and activate the virtual environment?",
    "What are the three types of tests?",
    "Which files are related to packaging?",
    "What commands build the package and what folders appear?",
]

THEME = gr.themes.Base(
    primary_hue="teal",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "ui-monospace", "monospace"],
)


def create_demo(pipeline: RAGPipeline) -> gr.Blocks:
    """Question box + streamed, cited answer + the passages it was grounded on."""

    def ask(question, search_type, k, lambda_mult):
        question = (question or "").strip()
        if not question:
            yield "Type a question about AST-1 or AST-2, or pick an example below.", ""
            return
        answer, docs = "", []
        try:
            for answer, docs in pipeline.stream(
                question, search_type=search_type, k=int(k), lambda_mult=float(lambda_mult)
            ):
                yield answer or "_Retrieving passages and drafting an answer…_", sources_markdown(docs)
        except Exception as exc:  # surface model / API errors in the UI
            raise gr.Error(f"Answer generation failed: {exc}") from exc
        missing = ungrounded_names(question, answer, docs)
        if missing:
            names = ", ".join(f"`{n}`" for n in missing)
            note = f"\n\n> ⚠️ Not found in the sources: {names}. Check the passages before relying on it."
            yield answer + note, sources_markdown(docs)

    with gr.Blocks(title="AST RAG Assistant") as demo:
        gr.Markdown(
            "## Ask the AST-1 & AST-2 guides\n"
            "Answers come only from the two assignment documents, with numbered citations "
            "to the passages used."
        )
        with gr.Row():
            with gr.Column(scale=3):
                question = gr.Textbox(
                    label="Your question",
                    placeholder="e.g. What does conftest.py do?",
                    lines=2,
                    max_length=500,
                )
                ask_btn = gr.Button("Get answer", variant="primary")
                gr.Markdown("### Answer")
                answer = gr.Markdown("Your answer will appear here, with [n] citations matching the sources.")
                gr.Examples(EXAMPLES, inputs=question)
            with gr.Column(scale=2):
                with gr.Accordion("Retrieval settings", open=False):
                    search_type = gr.Radio(
                        ["mmr", "similarity"],
                        value=pipeline.search_type,
                        label="Search type",
                        info="MMR picks diverse passages; similarity picks the closest ones.",
                    )
                    k = gr.Slider(1, 8, value=pipeline.k, step=1, label="Passages to retrieve (k)")
                    lambda_mult = gr.Slider(
                        0.0,
                        1.0,
                        value=pipeline.lambda_mult,
                        step=0.1,
                        label="MMR relevance vs diversity (λ)",
                        info="1.0 = pure relevance, 0.0 = maximum diversity",
                    )
                gr.Markdown("### Sources")
                sources = gr.Markdown("Retrieved passages will appear here.")

        inputs = [question, search_type, k, lambda_mult]
        ask_btn.click(ask, inputs, [answer, sources])
        question.submit(ask, inputs, [answer, sources])
    return demo

In [ ]:
import sys

# Detect if we are running in Google Colab
IN_COLAB = "google.colab" in sys.modules

demo = create_demo(pipeline)
demo.queue().launch(share=IN_COLAB, theme=THEME, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5620d6be373f2a4d3f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Deploy to Hugging Face Spaces (optional)

The `hf_space/` folder that ships with this notebook is a complete Gradio Space:

```
hf_space/
├── app.py            # Gradio UI (same create_demo as above) + start-up pipeline
├── rag_core.py       # the code in this notebook, as a module
├── rag_config.json   # written by the cell below: the configuration chosen above
├── requirements.txt
├── packages.txt      # tesseract-ocr (system package for the screenshot OCR)
├── README.md         # Space metadata (sdk: gradio)
├── data/AST-1.pdf, data/AST-2.pdf
└── tests/            # pytest suite (not uploaded)
```

**Manual route:** create a new Space (SDK *Gradio*, hardware *CPU basic*) and upload these files.
**Faster answers:** add a `GROQ_API_KEY` secret (*Settings → Variables and secrets*). The app then
uses Llama 3.1 8B through Groq automatically, and otherwise runs Qwen2.5-1.5B-Instruct on the CPU,
which works but is slow on the free tier.

**Programmatic route:** upload `hf_space.zip` to Colab, set `DEPLOY_TO_HF = True`, and run the cell below.

In [ ]:
import json
import shutil
import zipfile

DEPLOY_TO_HF = False
SPACE_ID = "ynagasupraja/ast-rag-assistant"

if Path("hf_space.zip").exists() and not Path("hf_space").exists():
    zipfile.ZipFile("hf_space.zip").extractall(".")

if Path("hf_space").exists():
    chosen = {
        "embedding": BEST["embedding"],
        "chunking": CHUNK_CONFIGS[BEST["chunking"]],
        "add_headers": BEST["headers"],
        "ocr_images": OCR_IMAGES,
        **{k: getattr(pipeline, k) for k in ("search_type", "k", "fetch_k", "lambda_mult")},
    }
    Path("hf_space/rag_config.json").write_text(json.dumps(chosen, indent=2))
    Path("hf_space/data").mkdir(exist_ok=True)
    for pdf in PDF_PATHS:
        shutil.copy(pdf, "hf_space/data/")
    print("Wrote hf_space/rag_config.json:", chosen)

if DEPLOY_TO_HF:
    from huggingface_hub import HfApi, login

    login()  # paste a Hugging Face token with write access
    api = HfApi()
    api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
    api.upload_folder(
        folder_path="hf_space",
        repo_id=SPACE_ID,
        repo_type="space",
        ignore_patterns=["tests/*", "**/__pycache__/*", "*.pyc", ".pytest_cache/*"],
    )
    print(f"Deployed: https://huggingface.co/spaces/{SPACE_ID}")

## Summary

* **Load:** PyMuPDF extraction, which avoids the split words pypdf produces on these files, plus OCR of the screenshots.
* **Chunk:** four strategies, page-break-aware merging with exact page citations, optional contextual headers.
* **Embed + store:** four open-source MTEB models with correct prefixes; FAISS for experiments, persistent Chroma for the system.
* **Generate:** grounded, cited answers from an open-source LLM, checked for citation-only replies and invented file names; MMR vs similarity compared on diversity and recall.
* **Evaluate:** tuned on one question set, reported on another, with an out-of-scope refusal check.
* **Serve:** streaming Gradio app, deployable to Hugging Face Spaces with the tuned configuration.

**Possible next steps:** hybrid BM25 + dense retrieval and a cross-encoder reranker (e.g. `BAAI/bge-reranker-base`).